In [1]:

import gc
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from scipy.stats import rankdata
from torch.utils.data import DataLoader
ROOT_DIR=Path(r'D:\MyJupyter\Works\FaceScore')
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0,str(ROOT_DIR))
from rating_dataset import RatingDataset
from model import SparseLandauScoreField
from others import create_baseline
TEST_DIR=ROOT_DIR/'data'/'test_face_data'
WEIGHTS_DIR=ROOT_DIR/'weights'
TABLE_DIR=ROOT_DIR/'outputs'/'tables'
MAIN_TABLE_FILE=TABLE_DIR/'main_comparison.csv'
GROUP_TABLE_FILE=TABLE_DIR/'group_comparison.csv'
METHODS=['ours','EUPE','TIPSV2','QWEN35','MOBILECLIP2','PECORE','IFORMER']
SEEDS=[42,123,2026,3407,777]
METRICS=['emd','mae','nll','spearman']
IMAGE_SIZE=336
IMAGE_BATCH_SIZE=4
HEAD_BATCH_SIZE=128
NUM_WORKERS=0
BOOTSTRAP_REPEATS=1000
CONFIDENCE_LEVEL=0.95
RNG_SEED=20261102
MAIN_MATCH_TOL=0.0005
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
if not TEST_DIR.is_dir():
    raise FileNotFoundError(f'测试集不存在: {TEST_DIR}')
if not MAIN_TABLE_FILE.is_file():
    raise FileNotFoundError(f'主对比表不存在: {MAIN_TABLE_FILE}')
if len(set(SEEDS))!=len(SEEDS) or len(SEEDS)<2:
    raise ValueError('SEEDS至少需要两个互不重复的训练种子')
if not 0.0<CONFIDENCE_LEVEL<1.0:
    raise ValueError('CONFIDENCE_LEVEL必须在(0,1)之间')
checkpoint_paths={(method,seed):WEIGHTS_DIR/f'{method}_{seed}.pt' for method in METHODS for seed in SEEDS}
missing_checkpoints=[str(path) for path in checkpoint_paths.values() if not path.is_file()]
if missing_checkpoints:
    raise FileNotFoundError('缺少训练权重:\n'+'\n'.join(missing_checkpoints))
main_table=pd.read_csv(MAIN_TABLE_FILE)
if set(main_table['model'].astype(str))!=set(METHODS):
    raise RuntimeError('主对比表模型列表与当前七模型不一致')
test_dataset=RatingDataset(str(TEST_DIR),mask_dir=None,augment=False,repeat=1,image_size=IMAGE_SIZE,soft_target=False)
test_loader=DataLoader(test_dataset,batch_size=IMAGE_BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=USE_AMP)
TEST_NAMES=[path.name for path in test_dataset.image_paths]
TEST_COUNTS=np.asarray([len(test_dataset._read_scores(path.with_suffix('.txt'))) for path in test_dataset.image_paths],dtype=np.int32)
if len(TEST_NAMES)!=len(set(TEST_NAMES)) or len(TEST_NAMES)!=len(TEST_COUNTS):
    raise RuntimeError('测试集照片文件名或评分人数不正确')
reference_row=main_table.loc[main_table['model']=='ours'].iloc[0]
if len(TEST_NAMES)!=int(reference_row['test_photos']) or int((TEST_COUNTS==1).sum())!=int(reference_row['single_rating_photos']):
    raise RuntimeError('当前测试集数量或单评分照片数量与main_comparison.csv不一致')
if str(reference_row['train_seeds'])!=','.join(map(str,SEEDS)):
    raise RuntimeError('当前训练种子与主对比表不一致')
print(f'测试照片: {len(TEST_NAMES)} | n=1: {(TEST_COUNTS==1).sum()} | n=2: {(TEST_COUNTS==2).sum()} | n>=3: {(TEST_COUNTS>=3).sum()}')
print(f'设备: {DEVICE} | 七个模型 | 五个训练随机种子 | 仅推理，不训练')


测试照片: 1480 | n=1: 507 | n=2: 340 | n>=3: 633
设备: cuda | 七个模型 | 五个训练随机种子 | 仅推理，不训练


In [2]:

class FrozenEncoderTokenAdapter(torch.nn.Module):
    def __init__(self,encoder,input_size,mean,std):
        super().__init__()
        self.encoder=encoder
        self.embed_dim=int(encoder.num_features)
        self.global_only=True
        self.input_size=int(input_size)
        self.register_buffer('mean',mean.detach().clone())
        self.register_buffer('std',std.detach().clone())
    def train(self,mode=True):
        super().train(mode)
        self.encoder.eval()
        return self
def build_model(method):
    if method!='ours':
        return create_baseline(method,pretrained=False,freeze_backbone=True,dropout=0.10)
    tip_model=create_baseline('TIPSV2',pretrained=False,freeze_backbone=True,dropout=0.0)
    adapter=FrozenEncoderTokenAdapter(tip_model.backbone,tip_model.input_size,tip_model.mean,tip_model.std)
    net=SparseLandauScoreField(pretrained=False,backbone=adapter,dropout=0.10,train_last_stage=False,width=24,prior_mix=0.0,use_entropy=False,use_high_resolution=False,scales=(224,),grid_side=16,use_evidence_head=True,evidence_base=0.1,use_global_token=True)
    del tip_model
    return net
def read_checkpoint(path):
    state=torch.load(path,map_location='cpu',weights_only=True)
    if not isinstance(state,dict) or not state or not all(isinstance(item,torch.Tensor) for item in state.values()):
        raise RuntimeError(f'权重不是直接保存的state_dict: {path}')
    return state
def verify_frozen_encoder(reference,current,method,seed):
    keys=[name for name in reference if name.startswith('backbone.') or name in ('mean','std')]
    mismatches=[name for name in keys if name not in current or not torch.equal(reference[name],current[name])]
    if mismatches:
        raise RuntimeError(f'{method}_{seed}的冻结视觉编码器与第一个种子不同: {mismatches[:5]}')
@torch.inference_mode()
def extract_features_and_labels(model,method):
    model.eval()
    feature_chunks=[]
    target_chunks=[]
    for images,targets in test_loader:
        images=images.to(DEVICE,non_blocking=True)
        with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=USE_AMP):
            features=model.encode_features(images) if method=='ours' else model.extract_features(images)
        feature_chunks.append(features.float().cpu())
        target_chunks.append(targets.float().cpu())
    return torch.cat(feature_chunks,dim=0),torch.cat(target_chunks,dim=0).numpy().astype(np.float64)
@torch.inference_mode()
def predict_cached_features(model,features):
    model.eval()
    predictions=[]
    for start in range(0,len(features),HEAD_BATCH_SIZE):
        batch=features[start:start+HEAD_BATCH_SIZE].to(DEVICE,non_blocking=True)
        with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=USE_AMP):
            output=model.forward_from_features(batch)
        log_probs=output[0] if isinstance(output,(tuple,list)) else output
        if log_probs.shape!=(len(batch),10):
            raise RuntimeError(f'评分分布输出形状错误: {tuple(log_probs.shape)}')
        predictions.append(torch.softmax(log_probs.float(),dim=1).cpu().numpy().astype(np.float64))
    result=np.concatenate(predictions,axis=0)
    if not np.isfinite(result).all() or (result<0).any() or not np.allclose(result.sum(axis=1),1.0,atol=1e-5):
        raise RuntimeError('预测不是合法的10维概率分布')
    return result


In [3]:

all_predictions={}
test_targets=None
for method in METHODS:
    print(f'开始提取冻结特征: {method}',flush=True)
    model=build_model(method)
    reference_state=read_checkpoint(checkpoint_paths[(method,SEEDS[0])])
    model.load_state_dict(reference_state,strict=True)
    model=model.to(DEVICE).eval()
    features,labels=extract_features_and_labels(model,method)
    if test_targets is None:
        test_targets=labels
    elif not np.array_equal(test_targets,labels):
        raise RuntimeError(f'{method}测试集标签不一致')
    seed_predictions=[]
    for seed in SEEDS:
        state=reference_state if seed==SEEDS[0] else read_checkpoint(checkpoint_paths[(method,seed)])
        if seed!=SEEDS[0]:
            verify_frozen_encoder(reference_state,state,method,seed)
            model.load_state_dict(state,strict=True)
        seed_predictions.append(predict_cached_features(model,features))
        print(f'已完成: {method}_{seed}.pt',flush=True)
        if seed!=SEEDS[0]:
            del state
    all_predictions[method]=np.stack(seed_predictions,axis=0)
    del model,reference_state,features,labels,seed_predictions
    gc.collect()
    if USE_AMP:
        torch.cuda.empty_cache()
if test_targets.shape!=(len(TEST_NAMES),10) or not np.isfinite(test_targets).all() or (test_targets<0).any() or not np.allclose(test_targets.sum(axis=1),1.0,atol=1e-5):
    raise RuntimeError('测试标签不是有效的[N,10]经验分布')
print('七模型×五种子预测全部完成')


开始提取冻结特征: ours
已完成: ours_42.pt
已完成: ours_123.pt
已完成: ours_2026.pt
已完成: ours_3407.pt
已完成: ours_777.pt
开始提取冻结特征: EUPE
已完成: EUPE_42.pt
已完成: EUPE_123.pt
已完成: EUPE_2026.pt
已完成: EUPE_3407.pt
已完成: EUPE_777.pt
开始提取冻结特征: TIPSV2
已完成: TIPSV2_42.pt
已完成: TIPSV2_123.pt
已完成: TIPSV2_2026.pt
已完成: TIPSV2_3407.pt
已完成: TIPSV2_777.pt
开始提取冻结特征: QWEN35
已完成: QWEN35_42.pt
已完成: QWEN35_123.pt
已完成: QWEN35_2026.pt
已完成: QWEN35_3407.pt
已完成: QWEN35_777.pt
开始提取冻结特征: MOBILECLIP2
已完成: MOBILECLIP2_42.pt
已完成: MOBILECLIP2_123.pt
已完成: MOBILECLIP2_2026.pt
已完成: MOBILECLIP2_3407.pt
已完成: MOBILECLIP2_777.pt
开始提取冻结特征: PECORE
已完成: PECORE_42.pt
已完成: PECORE_123.pt
已完成: PECORE_2026.pt
已完成: PECORE_3407.pt
已完成: PECORE_777.pt
开始提取冻结特征: IFORMER
已完成: IFORMER_42.pt
已完成: IFORMER_123.pt
已完成: IFORMER_2026.pt
已完成: IFORMER_3407.pt
已完成: IFORMER_777.pt
七模型×五种子预测全部完成


In [5]:

SCORE_BINS=np.arange(1,11,dtype=np.float64)
TARGET_MEAN=test_targets@SCORE_BINS
GROUPS={'Overall':np.ones(len(TEST_NAMES),dtype=bool),'n=1':TEST_COUNTS==1,'n=2':TEST_COUNTS==2,'n>=3':TEST_COUNTS>=3}
if sum(int(GROUPS[name].sum()) for name in ['n=1','n=2','n>=3'])!=len(TEST_NAMES):
    raise RuntimeError('评分人数分组没有完整覆盖测试集')
for group_name,mask in GROUPS.items():
    if int(mask.sum())<10 or np.unique(TARGET_MEAN[mask]).size<2:
        raise RuntimeError(f'{group_name}样本不足或者经验均分没有变化，不适合计算当前Spearman和CI')
def spearman_rows(x,y):
    x=np.asarray(x,dtype=np.float64)
    if x.ndim==1:
        x=x[None,:]
    rank_x=rankdata(x,axis=1)
    rank_y=rankdata(np.asarray(y,dtype=np.float64))
    rank_x=rank_x-rank_x.mean(axis=1,keepdims=True)
    rank_y=rank_y-rank_y.mean()
    denominator=np.linalg.norm(rank_x,axis=1)*np.linalg.norm(rank_y)
    return np.divide((rank_x*rank_y[None,:]).sum(axis=1),denominator,out=np.full(x.shape[0],np.nan),where=denominator>1e-12)
group_info={}
for group_name,mask in GROUPS.items():
    true_probs=test_targets[mask]
    true_means=TARGET_MEAN[mask]
    for method in METHODS:
        p=all_predictions[method][:,mask,:]
        cdf_delta=np.cumsum(p,axis=2)-np.cumsum(true_probs[None,:,:],axis=2)
        emd=np.sqrt(np.square(cdf_delta).mean(axis=2))
        pred_means=p@SCORE_BINS
        mae=np.abs(pred_means-true_means[None,:])
        nll=-(true_probs[None,:,:]*np.log(np.clip(p,1e-12,1.0))).sum(axis=2)
        additive=np.stack([emd,mae,nll],axis=2)
        rank_rho=spearman_rows(pred_means,true_means)
        per_seed=np.column_stack((additive.mean(axis=1),rank_rho))
        if not np.isfinite(per_seed).all():
            raise RuntimeError(f'{group_name} / {method}的指标存在NaN')
        group_info[(group_name,method)]={'additive':additive,'pred_mean':pred_means,'true_mean':true_means,'per_seed':per_seed,'mean':per_seed.mean(axis=0),'seed_sd':per_seed.std(axis=0,ddof=1)}
for method in METHODS:
    overall=group_info[('Overall',method)]['mean']
    reference=main_table.loc[main_table['model']==method].iloc[0]
    for metric_index,metric in enumerate(METRICS):
        stored=float(reference[f'{metric}_mean'])
        if abs(float(overall[metric_index])-stored)>MAIN_MATCH_TOL:
            raise RuntimeError(f'{method}整体{metric}与主对比表不一致: 当前={overall[metric_index]:.7f},主表={stored:.7f}')
print('整体指标已与main_comparison.csv中七种方法逐项核对通过')
for group_name,mask in GROUPS.items():
    print(f'{group_name}: {int(mask.sum())}张 | 原始评分均值={TARGET_MEAN[mask].mean():.4f}')
    for method in METHODS:
        values=group_info[(group_name,method)]['mean']
        print(f'{method:12s} EMD={values[0]:.5f} MAE={values[1]:.5f} NLL={values[2]:.5f} Spearman={values[3]:.5f}')


整体指标已与main_comparison.csv中七种方法逐项核对通过
Overall: 1480张 | 原始评分均值=4.8020
ours         EMD=0.22043 MAE=1.04521 NLL=1.78549 Spearman=0.64414
EUPE         EMD=0.22398 MAE=1.09541 NLL=1.79159 Spearman=0.55608
TIPSV2       EMD=0.22606 MAE=1.09844 NLL=1.80494 Spearman=0.54901
QWEN35       EMD=0.24982 MAE=1.30209 NLL=1.98510 Spearman=0.45913
MOBILECLIP2  EMD=0.23474 MAE=1.17568 NLL=1.83762 Spearman=0.45583
PECORE       EMD=0.22295 MAE=1.07373 NLL=1.78572 Spearman=0.54193
IFORMER      EMD=0.24606 MAE=1.24934 NLL=1.97573 Spearman=0.34046
n=1: 507张 | 原始评分均值=5.0592
ours         EMD=0.28213 MAE=1.34032 NLL=1.79149 Spearman=0.62104
EUPE         EMD=0.28272 MAE=1.37134 NLL=1.75998 Spearman=0.55213
TIPSV2       EMD=0.28136 MAE=1.36168 NLL=1.73990 Spearman=0.57898
QWEN35       EMD=0.31812 MAE=1.68213 NLL=2.10069 Spearman=0.48209
MOBILECLIP2  EMD=0.29145 MAE=1.45444 NLL=1.78641 Spearman=0.49165
PECORE       EMD=0.27455 MAE=1.31257 NLL=1.71649 Spearman=0.56239
IFORMER      EMD=0.31002 MAE=1.58628 NLL=1.99575

In [6]:

def sample_metrics(info,photo_ids,seed_ids):
    additive=info['additive'][seed_ids][:,photo_ids,:].mean(axis=(0,1))
    rho=spearman_rows(info['pred_mean'][seed_ids][:,photo_ids],info['true_mean'][photo_ids]).mean()
    return np.concatenate([additive,np.asarray([rho])])
bootstrap_results={}
for group_index,(group_name,mask) in enumerate(GROUPS.items()):
    rng=np.random.default_rng(RNG_SEED+group_index*1009)
    group_count=int(mask.sum())
    replicates=np.empty((BOOTSTRAP_REPEATS,len(METHODS),len(METRICS)),dtype=np.float64)
    for iteration in range(BOOTSTRAP_REPEATS):
        photo_ids=rng.integers(0,group_count,size=group_count)
        seed_ids=rng.integers(0,len(SEEDS),size=len(SEEDS))
        for method_index,method in enumerate(METHODS):
            replicates[iteration,method_index]=sample_metrics(group_info[(group_name,method)],photo_ids,seed_ids)
    valid=np.isfinite(replicates).all(axis=(1,2))
    replicates=replicates[valid]
    if len(replicates)<BOOTSTRAP_REPEATS*0.90:
        raise RuntimeError(f'{group_name}有效Bootstrap次数低于90%')
    bootstrap_results[group_name]=replicates
    print(f'{group_name}: 有效Bootstrap={len(replicates)}',flush=True)
rows=[]
alpha=(1.0-CONFIDENCE_LEVEL)/2.0
for group_name,mask in GROUPS.items():
    boot=bootstrap_results[group_name]
    group_size=int(mask.sum())
    for method_index,method in enumerate(METHODS):
        info=group_info[(group_name,method)]
        row={'group':group_name,'model':method,'photos':group_size,'rating_count_mean':float(TEST_COUNTS[mask].mean()),'ground_truth_score_mean':float(TARGET_MEAN[mask].mean()),'train_seed_count':len(SEEDS),'train_seeds':','.join(map(str,SEEDS)),'confidence_level':CONFIDENCE_LEVEL,'bootstrap_repeats':len(boot)}
        for metric_index,metric in enumerate(METRICS):
            row[f'{metric}_mean']=float(info['mean'][metric_index])
            row[f'{metric}_seed_sd']=float(info['seed_sd'][metric_index])
            row[f'{metric}_ci_low']=float(np.quantile(boot[:,method_index,metric_index],alpha))
            row[f'{metric}_ci_high']=float(np.quantile(boot[:,method_index,metric_index],1.0-alpha))
            if method=='ours':
                row[f'{metric}_ours_advantage']=np.nan
                row[f'{metric}_advantage_ci_low']=np.nan
                row[f'{metric}_advantage_ci_high']=np.nan
                row[f'{metric}_ours_better_seed_count']=np.nan
            else:
                sign=1.0 if metric=='spearman' else -1.0
                ours=group_info[(group_name,'ours')]
                row[f'{metric}_ours_advantage']=float(sign*(ours['mean'][metric_index]-info['mean'][metric_index]))
                advantage_boot=sign*(boot[:,0,metric_index]-boot[:,method_index,metric_index])
                row[f'{metric}_advantage_ci_low']=float(np.quantile(advantage_boot,alpha))
                row[f'{metric}_advantage_ci_high']=float(np.quantile(advantage_boot,1.0-alpha))
                row[f'{metric}_ours_better_seed_count']=int((sign*(ours['per_seed'][:,metric_index]-info['per_seed'][:,metric_index])>0).sum())
        rows.append(row)
result_table=pd.DataFrame.from_records(rows)
if len(result_table)!=len(GROUPS)*len(METHODS):
    raise RuntimeError('分组结果行数不正确')
TABLE_DIR.mkdir(parents=True,exist_ok=True)
result_table.to_csv(GROUP_TABLE_FILE,index=False,encoding='utf-8-sig',float_format='%.8f')
print(result_table[['group','model','photos','emd_mean','mae_mean','nll_mean','spearman_mean']].to_string(index=False,float_format=lambda value:f'{value:.5f}'))
print(f'已保存唯一分组表格: {GROUP_TABLE_FILE}')
print('mean=五个训练种子的指标均值；seed_sd=种子间样本标准差；CI=照片×训练种子的配对双层Bootstrap')
print('ours_advantage>0表示ours优于该方法；本表为预先指定的分组描述与区间估计，不对测试集进行调参')
info

Overall: 有效Bootstrap=1000
n=1: 有效Bootstrap=1000
n=2: 有效Bootstrap=1000
n>=3: 有效Bootstrap=1000
  group       model  photos  emd_mean  mae_mean  nll_mean  spearman_mean
Overall        ours    1480   0.22043   1.04521   1.78549        0.64414
Overall        EUPE    1480   0.22398   1.09541   1.79159        0.55608
Overall      TIPSV2    1480   0.22606   1.09844   1.80494        0.54901
Overall      QWEN35    1480   0.24982   1.30209   1.98510        0.45913
Overall MOBILECLIP2    1480   0.23474   1.17568   1.83762        0.45583
Overall      PECORE    1480   0.22295   1.07373   1.78572        0.54193
Overall     IFORMER    1480   0.24606   1.24934   1.97573        0.34046
    n=1        ours     507   0.28213   1.34032   1.79149        0.62104
    n=1        EUPE     507   0.28272   1.37134   1.75998        0.55213
    n=1      TIPSV2     507   0.28136   1.36168   1.73990        0.57898
    n=1      QWEN35     507   0.31812   1.68213   2.10069        0.48209
    n=1 MOBILECLIP2     507   0

{'additive': array([[[2.94497467e-01, 1.88819899e+00, 2.49292190e+00],
         [1.54967068e-01, 3.70987095e-01, 2.08576672e+00],
         [1.47745979e-01, 5.21929332e-01, 1.98283709e+00],
         ...,
         [2.21794446e-01, 8.60798255e-01, 1.80512865e+00],
         [2.43028783e-01, 1.40070315e+00, 1.59955555e+00],
         [2.01251813e-01, 7.22873227e-01, 1.88243892e+00]],
 
        [[2.89460010e-01, 1.85487500e+00, 2.38759163e+00],
         [1.57316084e-01, 6.69507453e-01, 1.97184930e+00],
         [1.09322862e-01, 8.63368390e-04, 1.76267234e+00],
         ...,
         [1.99928367e-01, 8.99005764e-01, 1.60090138e+00],
         [2.30885757e-01, 1.31550042e+00, 1.58056767e+00],
         [1.54732462e-01, 4.70248366e-01, 1.54747512e+00]],
 
        [[2.94003984e-01, 1.85060859e+00, 2.80354413e+00],
         [1.25496535e-01, 6.19848890e-01, 1.67885730e+00],
         [9.73741457e-02, 3.14517552e-02, 1.61097902e+00],
         ...,
         [1.62233481e-01, 6.69441943e-01, 1.43923183e+0